In [ ]:
from itertools import product
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from scaling.visualize import visualize_train_curves, plot_line_fit, plot_isoflops
from pathlib import Path

In [ ]:
unique_col_list = ["base_N", "target_N", "tkpm", "shrink", "method"]
y_col = "Validation Loss"
x_col = "flops"


def preprocess_warmstarting(df, y_col_to_smooth=None, smoothing_window=100):
    __df = pd.DataFrame()
    for i, x in enumerate(df.groupby(unique_col_list)):
        _df = x[1].sort_values(by="flops")
        # smooth it
        if y_col_to_smooth is not None:
            # +"_smoothed"
            _df[y_col_to_smooth] = _df[y_col_to_smooth].rolling(smoothing_window, win_type='gaussian', min_periods=1).mean(std=smoothing_window / 10)
        
        # scaling tokens and flops to the max
        max_intended_tokens = (_df.iloc[-1]["target_N"] * _df.iloc[-1]["tkpm"])
        if abs((max_intended_tokens -  _df["tokens"].max()) / _df["tokens"].max()) > 0.01:
            print("Wrong tkpm: ", x[0])
            continue
        _df["tokens"] = np.round(max_intended_tokens / _df["tokens"].max() * _df["tokens"])
        
        max_intended_flops = 6. * max_intended_tokens * _df["target_N"]
        _df["flops"] = np.round(max_intended_flops / _df["flops"].max() * _df["flops"])
        
        __df = pd.concat([__df, _df])
    
    print('Droping tkpm <= 5')    
    __df = __df[__df['tkpm'] > 5.]
    
    return __df

   
def get_loss_at_flops(df: pd.DataFrame, flop_intervals: list[float], y_col: str, unique_col_list = list[str], add_base_compute=False) -> pd.Series:
    """Get the loss at a specific flop value by interpolation."""
    x_col = "flops"
    best_learning_curve = None
    best_final_loss = float('inf')

    for i, x in enumerate(df.groupby(unique_col_list)):
        _df = x[1].dropna(subset=[y_col]).sort_values(by=x_col)
        if add_base_compute:
            base_flops = 6. * 20. * _df.iloc[0]['base_N']**2
            _df[x_col] += base_flops
        final_loss = _df.iloc[-1][y_col]
        if final_loss < best_final_loss:
            best_final_loss = final_loss
            best_learning_curve = pd.Series(
                data=_df[y_col].values,
                index=_df[x_col].values
            )
    
    # add the flops into the Series if not present
    for flop in flop_intervals:
        if flop not in best_learning_curve.index:
            best_learning_curve.loc[flop] = np.nan
    best_learning_curve = best_learning_curve.sort_index()
    # interpolate nans
    best_learning_curve = best_learning_curve.interpolate(method='linear')
    return best_learning_curve.loc[flop_intervals]


In [ ]:
warmstarting_df = pd.read_parquet(
    "../data/warmstart_runs_flattened.parquet",
)
warmstarting_df = warmstarting_df.dropna(subset=[y_col])
warmstarting_df = preprocess_warmstarting(warmstarting_df)

In [ ]:
warmstarting_df['method'].unique()

In [ ]:
test_df = pd.read_parquet(
    "../data/warmstart_runs_flattened.parquet",
)

# select method mup with the largest tokens available and the max steps and flops
# warmstarting_df = warmstarting_df[warmstarting_df['method'] == 'mup']
test_df = test_df[test_df['target_N'] == 1217722368]
test_df = test_df.sort_values(by=['flops'], ascending=[False])
# warmstarting_df.dropna(subset=[y_col])

In [ ]:
test_df['tokens'] / test_df['target_N']

In [ ]:
import seaborn as sns
sns.set_style("white")
sns.set_context("notebook", font_scale=1.5)

ADD_BASE_COMPUTE = False
TKPM = 10.
MIN_FLOPS_SCALE_FACTOR = 6
NUM_FLOP_INTERVALS = 8
METHOD = 'net2net'
SHRINK = None
TARGET_RANGE_L = 1
TARGET_RANGE_R = 5

warmstarting_tkpms_df = warmstarting_df[warmstarting_df['tkpm']==TKPM]
method_tkpms_df = warmstarting_tkpms_df[warmstarting_tkpms_df['method']==METHOD]

target_models = sorted(method_tkpms_df['target_N'].unique())
target_models = target_models[TARGET_RANGE_L:TARGET_RANGE_R]  # skip the smallest model
fig, axes = plt.subplots(1, len(target_models), figsize=(5 * len(target_models), 5), layout='constrained')
axes = np.atleast_1d(axes)

tables = {}
for i, target_model in enumerate(target_models):
    target_model_df = warmstarting_tkpms_df[warmstarting_tkpms_df['target_N']==target_model]
    no_growth_df = target_model_df[target_model_df['method']=='mup']
    
    # calculate flop intervals
    max_flops = no_growth_df['flops'].max()
    min_flops = max_flops / MIN_FLOPS_SCALE_FACTOR
    flop_intervals = np.linspace(min_flops, max_flops, NUM_FLOP_INTERVALS)
    flops_df = pd.DataFrame()
    
    # add growth factor == 1
    no_growth_df = no_growth_df[no_growth_df["base_N"]==no_growth_df["base_N"].max()]
    flops_df[1.] = get_loss_at_flops(no_growth_df, flop_intervals, y_col, unique_col_list)
    
    # add growth factor > 1
    shrink_target_model_df = target_model_df[target_model_df['method'] == METHOD]
    if SHRINK is not None:
        shrink_target_model_df = shrink_target_model_df[shrink_target_model_df['shrink']==SHRINK]
    # check if it is in a shrink list
    
    base_models = sorted(shrink_target_model_df['base_N'].unique(), reverse=True)
    for base_model in base_models:
        base_model_df = shrink_target_model_df[shrink_target_model_df['base_N']==base_model]
        
        growth_factor = target_model / base_model
        growth_df = base_model_df[base_model_df['target_N']==target_model]
        flops_df[growth_factor] = get_loss_at_flops(growth_df, flop_intervals, y_col, unique_col_list, add_base_compute=ADD_BASE_COMPUTE)
        # select only the shrink factor we want
    axes[i].set_title(f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}")
    table_df = plot_isoflops(
        axes[i],
        flops_df,
        disable_y_label=(i == len(target_models) - 1),
        return_table=True,
    )
    tables[f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}"] = table_df
    
# add figure wide xlabel
fig.supxlabel("Growth Factor")
fig.supylabel(y_col)
fig.suptitle(f"{METHOD} - GPT2")
# save the figure in a great resulution
fig.savefig(f"figures/neurips_{METHOD}_{TKPM}-tkpm_isoflops_.pdf")

# create a markdown file for each table
for title, table_df in tables.items():
    table_df.index = table_df.index.map(lambda x: f"{x:.1e}")
    for col in table_df.columns:
        table_df[col] = table_df[col].map(lambda x: f"{x:.1f}")
                                          
    Path(f"tables/rebuttal_isoflops_net2net/{title}.md").parent.mkdir(parents=True, exist_ok=True)
    table_df.to_markdown(f"tables/rebuttal_isoflops_net2net/{title}.md")


In [ ]:
import seaborn as sns
sns.set_style("white")
sns.set_context("notebook", font_scale=1.5)

ADD_BASE_COMPUTE = False
TKPM = 20.
MIN_FLOPS_SCALE_FACTOR = 6
NUM_FLOP_INTERVALS = 8
METHOD = 'net2net'
SHRINK = None
TARGET_RANGE_L = 1
TARGET_RANGE_R = 4

warmstarting_tkpms_df = warmstarting_df[warmstarting_df['tkpm']==TKPM]
method_tkpms_df = warmstarting_tkpms_df[warmstarting_tkpms_df['method']==METHOD]

target_models = sorted(method_tkpms_df['target_N'].unique())
target_models = target_models[TARGET_RANGE_L:TARGET_RANGE_R]  # skip the smallest model
fig, axes = plt.subplots(1, len(target_models), figsize=(5 * len(target_models), 5), layout='constrained')
axes = np.atleast_1d(axes)

tables = {}
for i, target_model in enumerate(target_models):
    target_model_df = warmstarting_tkpms_df[warmstarting_tkpms_df['target_N']==target_model]
    no_growth_df = target_model_df[target_model_df['method']=='mup']

    # calculate flop intervals
    max_flops = no_growth_df['flops'].max()
    min_flops = max_flops / MIN_FLOPS_SCALE_FACTOR
    flop_intervals = np.linspace(min_flops, max_flops, NUM_FLOP_INTERVALS)
    flops_df = pd.DataFrame()

    # add growth factor == 1
    no_growth_df = no_growth_df[no_growth_df["base_N"]==no_growth_df["base_N"].max()]
    flops_df[1.] = get_loss_at_flops(no_growth_df, flop_intervals, y_col, unique_col_list)

    # add growth factor > 1
    shrink_target_model_df = target_model_df[target_model_df['method'] == METHOD]
    if SHRINK is not None:
        shrink_target_model_df = shrink_target_model_df[shrink_target_model_df['shrink']==SHRINK]
    # check if it is in a shrink list

    base_models = sorted(shrink_target_model_df['base_N'].unique(), reverse=True)
    for base_model in base_models:
        base_model_df = shrink_target_model_df[shrink_target_model_df['base_N']==base_model]

        growth_factor = target_model / base_model
        growth_df = base_model_df[base_model_df['target_N']==target_model]
        flops_df[growth_factor] = get_loss_at_flops(growth_df, flop_intervals, y_col, unique_col_list, add_base_compute=ADD_BASE_COMPUTE)
        # select only the shrink factor we want
    axes[i].set_title(f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}")
    table_df = plot_isoflops(
        axes[i],
        flops_df,
        disable_y_label=(i != len(target_models) - 1),
        return_table=True,
    )
    tables[f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}"] = table_df

# add figure wide xlabel
fig.supxlabel("Growth Factor")
fig.supylabel(y_col)
fig.suptitle(f"{METHOD} - GPT2")
# save the figure in a great resulution
fig.savefig(f"figures/neurips_{METHOD}_{TKPM}-tkpm_isoflops_.pdf")

# create a markdown file for each table
for title, table_df in tables.items():
    table_df.index = table_df.index.map(lambda x: f"{x:.1e}")
    for col in table_df.columns:
        table_df[col] = table_df[col].map(lambda x: f"{x:.1f}")

    Path(f"tables/rebuttal_isoflops_net2net/{title}.md").parent.mkdir(parents=True, exist_ok=True)
    table_df.to_markdown(f"tables/rebuttal_isoflops_net2net/{title}.md")


In [ ]:
import seaborn as sns
sns.set_style("white")
sns.set_context("notebook", font_scale=1.5)

ADD_BASE_COMPUTE = False
TKPM = 30.
MIN_FLOPS_SCALE_FACTOR = 6
NUM_FLOP_INTERVALS = 8
METHOD = 'net2net'
SHRINK = None
TARGET_RANGE_L = 1
TARGET_RANGE_R = 10

warmstarting_tkpms_df = warmstarting_df[warmstarting_df['tkpm']==TKPM]
method_tkpms_df = warmstarting_tkpms_df[warmstarting_tkpms_df['method']==METHOD]

target_models = sorted(method_tkpms_df['target_N'].unique())
target_models = target_models[TARGET_RANGE_L:TARGET_RANGE_R]  # skip the smallest model
fig, axes = plt.subplots(1, len(target_models), figsize=(5 * len(target_models), 5), layout='constrained')
axes = np.atleast_1d(axes)

tables = {}
for i, target_model in enumerate(target_models):
    target_model_df = warmstarting_tkpms_df[warmstarting_tkpms_df['target_N']==target_model]
    no_growth_df = target_model_df[target_model_df['method']=='mup']

    # calculate flop intervals
    max_flops = no_growth_df['flops'].max()
    min_flops = max_flops / MIN_FLOPS_SCALE_FACTOR
    flop_intervals = np.linspace(min_flops, max_flops, NUM_FLOP_INTERVALS)
    flops_df = pd.DataFrame()

    # add growth factor == 1
    no_growth_df = no_growth_df[no_growth_df["base_N"]==no_growth_df["base_N"].max()]
    flops_df[1.] = get_loss_at_flops(no_growth_df, flop_intervals, y_col, unique_col_list)

    # add growth factor > 1
    shrink_target_model_df = target_model_df[target_model_df['method'] == METHOD]
    if SHRINK is not None:
        shrink_target_model_df = shrink_target_model_df[shrink_target_model_df['shrink']==SHRINK]
    # check if it is in a shrink list

    base_models = sorted(shrink_target_model_df['base_N'].unique(), reverse=True)
    for base_model in base_models:
        base_model_df = shrink_target_model_df[shrink_target_model_df['base_N']==base_model]

        growth_factor = target_model / base_model
        growth_df = base_model_df[base_model_df['target_N']==target_model]
        flops_df[growth_factor] = get_loss_at_flops(growth_df, flop_intervals, y_col, unique_col_list, add_base_compute=ADD_BASE_COMPUTE)
        # select only the shrink factor we want
    axes[i].set_title(f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}")
    table_df = plot_isoflops(
        axes[i],
        flops_df,
        disable_y_label=(i == len(target_models) - 1),
        return_table=True,
    )
    tables[f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}"] = table_df

# add figure wide xlabel
fig.supxlabel("Growth Factor")
fig.supylabel(y_col)
fig.suptitle(f"{METHOD} - GPT2")
# save the figure in a great resulution
fig.savefig(f"figures/neurips_{METHOD}_{TKPM}-tkpm_isoflops_.pdf")

# create a markdown file for each table
for title, table_df in tables.items():
    table_df.index = table_df.index.map(lambda x: f"{x:.1e}")
    for col in table_df.columns:
        table_df[col] = table_df[col].map(lambda x: f"{x:.1f}")

    Path(f"tables/rebuttal_isoflops_net2net/{title}.md").parent.mkdir(parents=True, exist_ok=True)
    table_df.to_markdown(f"tables/rebuttal_isoflops_net2net/{title}.md")

In [ ]:
import seaborn as sns
sns.set_style("white")
sns.set_context("notebook", font_scale=1.5)

ADD_BASE_COMPUTE = False
TKPM = 30.
MIN_FLOPS_SCALE_FACTOR = 6
NUM_FLOP_INTERVALS = 8
METHOD = 'paws'
SHRINK = 0.4
TARGET_RANGE_L = 1
TARGET_RANGE_R = 4

warmstarting_tkpms_df = warmstarting_df[warmstarting_df['tkpm']==TKPM]
method_tkpms_df = warmstarting_tkpms_df[warmstarting_tkpms_df['method']==METHOD]

target_models = sorted(method_tkpms_df['target_N'].unique())
target_models = target_models[TARGET_RANGE_L:TARGET_RANGE_R]  # skip the smallest model
fig, axes = plt.subplots(1, len(target_models), figsize=(5 * len(target_models), 5), layout='constrained')
axes = np.atleast_1d(axes)

tables = {}
for i, target_model in enumerate(target_models):
    target_model_df = warmstarting_tkpms_df[warmstarting_tkpms_df['target_N']==target_model]
    no_growth_df = target_model_df[target_model_df['method']=='mup']

    # calculate flop intervals
    max_flops = no_growth_df['flops'].max()
    min_flops = max_flops / MIN_FLOPS_SCALE_FACTOR
    flop_intervals = np.linspace(min_flops, max_flops, NUM_FLOP_INTERVALS)
    flops_df = pd.DataFrame()

    # add growth factor == 1
    no_growth_df = no_growth_df[no_growth_df["base_N"]==no_growth_df["base_N"].max()]
    flops_df[1.] = get_loss_at_flops(no_growth_df, flop_intervals, y_col, unique_col_list)

    # add growth factor > 1
    shrink_target_model_df = target_model_df[target_model_df['method'] == METHOD]
    if SHRINK is not None:
        shrink_target_model_df = shrink_target_model_df[shrink_target_model_df['shrink']==SHRINK]
    # check if it is in a shrink list

    base_models = sorted(shrink_target_model_df['base_N'].unique(), reverse=True)
    for base_model in base_models:
        base_model_df = shrink_target_model_df[shrink_target_model_df['base_N']==base_model]

        growth_factor = target_model / base_model
        growth_df = base_model_df[base_model_df['target_N']==target_model]
        flops_df[growth_factor] = get_loss_at_flops(growth_df, flop_intervals, y_col, unique_col_list, add_base_compute=ADD_BASE_COMPUTE)
        # select only the shrink factor we want
    axes[i].set_title(f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}")
    table_df = plot_isoflops(
        axes[i],
        flops_df,
        disable_y_label=(i == len(target_models) - 1),
        return_table=True,
    )
    tables[f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}"] = table_df

# add figure wide xlabel
fig.supxlabel("Growth Factor")
fig.supylabel(y_col)
fig.suptitle(f"{METHOD} - GPT2")
# save the figure in a great resulution
fig.savefig(f"figures/neurips_{METHOD}_{TKPM}-tkpm_isoflops_.pdf")

# create a markdown file for each table
for title, table_df in tables.items():
    table_df.index = table_df.index.map(lambda x: f"{x:.1e}")
    for col in table_df.columns:
        table_df[col] = table_df[col].map(lambda x: f"{x:.1f}")

    Path(f"tables/rebuttal_isoflops_net2net/{title}.md").parent.mkdir(parents=True, exist_ok=True)
    table_df.to_markdown(f"tables/rebuttal_isoflops_net2net/{title}.md")

In [ ]:
import seaborn as sns
sns.set_style("white")
sns.set_context("notebook", font_scale=1.5)

ADD_BASE_COMPUTE = False
TKPM = 20.
MIN_FLOPS_SCALE_FACTOR = 6
NUM_FLOP_INTERVALS = 8
METHOD = 'paws'
SHRINK = 0.4
TARGET_RANGE_L = 1
TARGET_RANGE_R = 4

warmstarting_tkpms_df = warmstarting_df[warmstarting_df['tkpm']==TKPM]
method_tkpms_df = warmstarting_tkpms_df[warmstarting_tkpms_df['method']==METHOD]

target_models = sorted(method_tkpms_df['target_N'].unique())
target_models = target_models[TARGET_RANGE_L:TARGET_RANGE_R]  # skip the smallest model
fig, axes = plt.subplots(1, len(target_models), figsize=(5 * len(target_models), 5), layout='constrained')
axes = np.atleast_1d(axes)

tables = {}
for i, target_model in enumerate(target_models):
    target_model_df = warmstarting_tkpms_df[warmstarting_tkpms_df['target_N']==target_model]
    no_growth_df = target_model_df[target_model_df['method']=='mup']

    # calculate flop intervals
    max_flops = no_growth_df['flops'].max()
    min_flops = max_flops / MIN_FLOPS_SCALE_FACTOR
    flop_intervals = np.linspace(min_flops, max_flops, NUM_FLOP_INTERVALS)
    flops_df = pd.DataFrame()

    # add growth factor == 1
    no_growth_df = no_growth_df[no_growth_df["base_N"]==no_growth_df["base_N"].max()]
    flops_df[1.] = get_loss_at_flops(no_growth_df, flop_intervals, y_col, unique_col_list)

    # add growth factor > 1
    shrink_target_model_df = target_model_df[target_model_df['method'] == METHOD]
    if SHRINK is not None:
        shrink_target_model_df = shrink_target_model_df[shrink_target_model_df['shrink']==SHRINK]
    # check if it is in a shrink list

    base_models = sorted(shrink_target_model_df['base_N'].unique(), reverse=True)
    for base_model in base_models:
        base_model_df = shrink_target_model_df[shrink_target_model_df['base_N']==base_model]

        growth_factor = target_model / base_model
        growth_df = base_model_df[base_model_df['target_N']==target_model]
        flops_df[growth_factor] = get_loss_at_flops(growth_df, flop_intervals, y_col, unique_col_list, add_base_compute=ADD_BASE_COMPUTE)
        # select only the shrink factor we want
    axes[i].set_title(f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}")
    table_df = plot_isoflops(
        axes[i],
        flops_df,
        disable_y_label=(i == len(target_models) - 1),
        return_table=True,
    )
    tables[f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}"] = table_df

# add figure wide xlabel
fig.supxlabel("Growth Factor", fontsize=15)
fig.supylabel(y_col, fontsize=15)
fig.suptitle(f"{METHOD} - GPT2", fontsize=20)
# save the figure in a great resulution
fig.savefig(f"figures/neurips_{METHOD}_{TKPM}-tkpm_isoflops_.pdf")

# create a markdown file for each table
for title, table_df in tables.items():
    table_df.index = table_df.index.map(lambda x: f"{x:.1e}")
    for col in table_df.columns:
        table_df[col] = table_df[col].map(lambda x: f"{x:.1f}")

    Path(f"tables/rebuttal_isoflops_net2net/{title}.md").parent.mkdir(parents=True, exist_ok=True)
    table_df.to_markdown(f"tables/rebuttal_isoflops_net2net/{title}.md")

In [ ]:
import seaborn as sns
sns.set_style("white")
sns.set_context("notebook", font_scale=1.5)

ADD_BASE_COMPUTE = False
TKPM = 10.
MIN_FLOPS_SCALE_FACTOR = 6
NUM_FLOP_INTERVALS = 8
METHOD = 'paws'
SHRINK = 0.4
TARGET_RANGE_L = 1
TARGET_RANGE_R = 10

warmstarting_tkpms_df = warmstarting_df[warmstarting_df['tkpm']==TKPM]
method_tkpms_df = warmstarting_tkpms_df[warmstarting_tkpms_df['method']==METHOD]

target_models = sorted(method_tkpms_df['target_N'].unique())
target_models = target_models[TARGET_RANGE_L:TARGET_RANGE_R]  # skip the smallest model
fig, axes = plt.subplots(1, len(target_models), figsize=(5 * len(target_models), 5), layout='constrained')
axes = np.atleast_1d(axes)

tables = {}
for i, target_model in enumerate(target_models):
    target_model_df = warmstarting_tkpms_df[warmstarting_tkpms_df['target_N']==target_model]
    no_growth_df = target_model_df[target_model_df['method']=='mup']

    # calculate flop intervals
    max_flops = no_growth_df['flops'].max()
    min_flops = max_flops / MIN_FLOPS_SCALE_FACTOR
    flop_intervals = np.linspace(min_flops, max_flops, NUM_FLOP_INTERVALS)
    flops_df = pd.DataFrame()

    # add growth factor == 1
    no_growth_df = no_growth_df[no_growth_df["base_N"]==no_growth_df["base_N"].max()]
    flops_df[1.] = get_loss_at_flops(no_growth_df, flop_intervals, y_col, unique_col_list)

    # add growth factor > 1
    shrink_target_model_df = target_model_df[target_model_df['method'] == METHOD]
    if SHRINK is not None:
        shrink_target_model_df = shrink_target_model_df[shrink_target_model_df['shrink']==SHRINK]
    # check if it is in a shrink list

    base_models = sorted(shrink_target_model_df['base_N'].unique(), reverse=True)
    for base_model in base_models:
        base_model_df = shrink_target_model_df[shrink_target_model_df['base_N']==base_model]

        growth_factor = target_model / base_model
        growth_df = base_model_df[base_model_df['target_N']==target_model]
        flops_df[growth_factor] = get_loss_at_flops(growth_df, flop_intervals, y_col, unique_col_list, add_base_compute=ADD_BASE_COMPUTE)
        # select only the shrink factor we want
    axes[i].set_title(f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}")
    table_df = plot_isoflops(
        axes[i],
        flops_df,
        disable_y_label=(i == len(target_models) - 1),
        return_table=True,
    )
    tables[f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}"] = table_df

# add figure wide xlabel
fig.supxlabel("Growth Factor", fontsize=15)
fig.supylabel(y_col, fontsize=15)
fig.suptitle(f"{METHOD} - GPT2", fontsize=20)
# save the figure in a great resulution
fig.savefig(f"figures/neurips_{METHOD}_{TKPM}-tkpm_isoflops.pdf")

# create a markdown file for each table
for title, table_df in tables.items():
    table_df.index = table_df.index.map(lambda x: f"{x:.1e}")
    for col in table_df.columns:
        table_df[col] = table_df[col].map(lambda x: f"{x:.1f}")

    Path(f"tables/rebuttal_isoflops_net2net/{title}.md").parent.mkdir(parents=True, exist_ok=True)
    table_df.to_markdown(f"tables/rebuttal_isoflops_net2net/{title}.md")

In [ ]:
# Fina

# Final Plots

In [ ]:
SHARED_WIDTH = 5.75
SHARED_HEIGHT = 5

## MLP Net2Net

In [ ]:
mlp_df = pd.read_parquet(
    "../data/mlp_results.parquet",
)
width_scale_to_params = {
    0: 0.1e6,
    1: 0.1e6,
    2: 0.2e6,
    3: 0.4e6,
    4: 0.9e6,
    5: 1.8e6,
    6: 3.5e6,
    7: 7.0e6,
    8: 14.2e6,
    9: 25.2e6,
    10: 100.7e6,
}
width_scale_to_hidden_dim = {
    0: 48,
    1: 68,
    2: 96,
    3: 136,
    4: 192,
    5: 272,
    6: 384,
    7: 540,
    8: 768,
    9: 1024,
    10: 2048,
}

# create base_N and target_N columns
mlp_df['base_N'] = mlp_df['base_scale'].map(lambda x: width_scale_to_hidden_dim.get(x, np.nan))
mlp_df['target_N'] = mlp_df['target_scale'].map(lambda x: width_scale_to_hidden_dim.get(x, np.nan))
mlp_df['max_flops'] = mlp_df['curve_flops'].map(lambda x: x[-1])

def get_loss_at_flops_mlp(df: pd.DataFrame, flop_intervals: list[float]) -> pd.Series:
    """Get the loss at a specific flop value by interpolation."""
    best_learning_curve = None
    best_final_loss = float('inf')

    # iterate over rows
    for row in df.itertuples(index=False):
        final_loss = row.curve_val[-1]
        if final_loss < best_final_loss:
            best_final_loss = final_loss
            best_learning_curve = pd.Series(
                data=row.curve_val,
                index=row.curve_flops
            )

    # add the flops into the Series if not present
    for flop in flop_intervals:
        if flop not in best_learning_curve.index:
            best_learning_curve.loc[flop] = np.nan
    best_learning_curve = best_learning_curve.sort_index()
    # interpolate nans
    best_learning_curve = best_learning_curve.interpolate(method='linear')
    return best_learning_curve.loc[flop_intervals]

mlp_df = mlp_df[mlp_df['scaling'] == 'width']
# mlp_df["target_N"] = mlp_df["target_params_m"] * 10 ** 6
# mlp_df["base_N"] = mlp_df["base_params_m"] * 10 ** 6

In [ ]:
mlp_df

In [ ]:
TKPM = 20.
MIN_FLOPS_SCALE_FACTOR = 6
NUM_FLOP_INTERVALS = 8
SCALING = "width"
METHOD = "net2net"


mlp_selection_df = mlp_df[mlp_df['tkpm_group']==TKPM]
mlp_selection_df = mlp_selection_df[mlp_selection_df['scaling']==SCALING]

target_models = sorted(mlp_selection_df['target_N'].unique())[5:]# [-3:-1]
fig, axes = plt.subplots(len(target_models), 1, figsize=(SHARED_WIDTH, SHARED_HEIGHT * len(target_models)), layout='constrained')
axes = np.atleast_1d(axes)
for i, target_model in enumerate(target_models):
    target_model_df = mlp_selection_df[mlp_selection_df['target_N']==target_model]
    no_growth_df = target_model_df[target_model_df['method']=='scratch']

    # calculate flop intervals
    max_flops = no_growth_df['max_flops'].max()
    min_flops = max_flops / MIN_FLOPS_SCALE_FACTOR
    flop_intervals = np.linspace(min_flops, max_flops, NUM_FLOP_INTERVALS)
    flops_df = pd.DataFrame()

    # add growth factor == 1
    flops_df[1.] = get_loss_at_flops_mlp(no_growth_df, flop_intervals)

    # add growth factor > 1
    shrink_target_model_df = target_model_df[target_model_df['method'] == METHOD]
    # check if it is in a shrink list

    base_models = sorted(shrink_target_model_df['base_N'].unique(), reverse=True)
    for base_model in base_models:
        base_model_df = shrink_target_model_df[shrink_target_model_df['base_N']==base_model]

        growth_factor = target_model / base_model
        growth_df = base_model_df[base_model_df['target_N']==target_model]
        flops_df[growth_factor] = get_loss_at_flops_mlp(growth_df, flop_intervals)
        # select only the shrink factor we want
    # axes[i].set_title(f"Target N: {(target_model/1_000_000):3.1f}M")
    axes[i].set_title(f"Width {target_model}, {TKPM} tkpm")
    plot_isoflops(
        axes[i],
        flops_df,
        disable_y_label=False,
    )

# add figure wide xlabel
fig.suptitle(f"{METHOD} - MLP")
fig.supxlabel(" ")
fig.supylabel(" ")
plt.show()
fig.savefig(f"figures/neurips_main_{METHOD}_{TKPM}-tkpm_isoflops_mlp.pdf")

In [ ]:
TKPM = 20.
MIN_FLOPS_SCALE_FACTOR = 6
NUM_FLOP_INTERVALS = 8
SCALING = "width"
METHOD = "snp"


mlp_selection_df = mlp_df[mlp_df['tkpm_group']==TKPM]
mlp_selection_df = mlp_selection_df[mlp_selection_df['scaling']==SCALING]

target_models = sorted(mlp_selection_df['target_N'].unique())[5:]# [-3:-1]
fig, axes = plt.subplots(len(target_models), 1, figsize=(SHARED_WIDTH, SHARED_HEIGHT * len(target_models)), layout='constrained')
axes = np.atleast_1d(axes)
for i, target_model in enumerate(target_models):
    target_model_df = mlp_selection_df[mlp_selection_df['target_N']==target_model]
    no_growth_df = target_model_df[target_model_df['method']=='scratch']

    # calculate flop intervals
    max_flops = no_growth_df['max_flops'].max()
    min_flops = max_flops / MIN_FLOPS_SCALE_FACTOR
    flop_intervals = np.linspace(min_flops, max_flops, NUM_FLOP_INTERVALS)
    flops_df = pd.DataFrame()

    # add growth factor == 1
    flops_df[1.] = get_loss_at_flops_mlp(no_growth_df, flop_intervals)

    # add growth factor > 1
    shrink_target_model_df = target_model_df[target_model_df['method'] == METHOD]
    # check if it is in a shrink list

    base_models = sorted(shrink_target_model_df['base_N'].unique(), reverse=True)
    for base_model in base_models:
        base_model_df = shrink_target_model_df[shrink_target_model_df['base_N']==base_model]

        growth_factor = target_model / base_model
        growth_df = base_model_df[base_model_df['target_N']==target_model] # Should be useless
        flops_df[growth_factor] = get_loss_at_flops_mlp(growth_df, flop_intervals)
        # select only the shrink factor we want
    # axes[i].set_title(f"Target N: {(target_model/1_000_000):3.1f}M")
    axes[i].set_title(f"Width {target_model}, {TKPM} tkpm")
    plot_isoflops(
        axes[i],
        flops_df,
        disable_y_label=False,
    )

# add figure wide xlabel
fig.suptitle(f"SZP - MLP")
fig.supxlabel(" ")
fig.supylabel(" ")
plt.show()
fig.savefig(f"figures/neurips_main_{METHOD}_{TKPM}-tkpm_isoflops_mlp.pdf")

## LLM Net2Net

In [ ]:
import seaborn as sns
sns.set_style("white")


ADD_BASE_COMPUTE = False
TKPM = 20.
MIN_FLOPS_SCALE_FACTOR = 6
NUM_FLOP_INTERVALS = 8
METHOD = 'net2net'
SHRINK = None
TARGET_RANGE_L = 2
TARGET_RANGE_R = 4

warmstarting_tkpms_df = warmstarting_df[warmstarting_df['tkpm']==TKPM]
method_tkpms_df = warmstarting_tkpms_df[warmstarting_tkpms_df['method']==METHOD]

target_models = sorted(method_tkpms_df['target_N'].unique())
target_models = target_models[TARGET_RANGE_L:TARGET_RANGE_R]  # skip the smallest model
fig, axes = plt.subplots(len(target_models), 1, figsize=(SHARED_WIDTH, SHARED_HEIGHT * len(target_models)), layout='constrained')
axes = np.atleast_1d(axes)

tables = {}
for i, target_model in enumerate(target_models):
    target_model_df = warmstarting_tkpms_df[warmstarting_tkpms_df['target_N']==target_model]
    no_growth_df = target_model_df[target_model_df['method']=='mup']

    # calculate flop intervals
    max_flops = no_growth_df['flops'].max()
    min_flops = max_flops / MIN_FLOPS_SCALE_FACTOR
    flop_intervals = np.linspace(min_flops, max_flops, NUM_FLOP_INTERVALS)
    flops_df = pd.DataFrame()

    # add growth factor == 1
    no_growth_df = no_growth_df[no_growth_df["base_N"]==no_growth_df["base_N"].max()]
    flops_df[1.] = get_loss_at_flops(no_growth_df, flop_intervals, y_col, unique_col_list)

    # add growth factor > 1
    shrink_target_model_df = target_model_df[target_model_df['method'] == METHOD]
    if SHRINK is not None:
        shrink_target_model_df = shrink_target_model_df[shrink_target_model_df['shrink']==SHRINK]
    # check if it is in a shrink list

    base_models = sorted(shrink_target_model_df['base_N'].unique(), reverse=True)
    for base_model in base_models:
        base_model_df = shrink_target_model_df[shrink_target_model_df['base_N']==base_model]

        growth_factor = target_model / base_model
        growth_df = base_model_df[base_model_df['target_N']==target_model]
        flops_df[growth_factor] = get_loss_at_flops(growth_df, flop_intervals, y_col, unique_col_list, add_base_compute=ADD_BASE_COMPUTE)
        # select only the shrink factor we want
    axes[i].set_title(f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}")
    table_df = plot_isoflops(
        axes[i],
        flops_df,
        disable_y_label=True,
        return_table=True,
    )
    tables[f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}"] = table_df

# add figure wide xlabel
fig.supxlabel(" ")
fig.supylabel(" ")
fig.supxlabel("Growth Factor")
fig.suptitle(f"{METHOD} - LLM")
# save the figure in a great resulution
fig.savefig(f"figures/neurips_main_{METHOD}_{TKPM}-tkpm_isoflops_gpt.pdf")

# create a markdown file for each table
for title, table_df in tables.items():
    table_df.index = table_df.index.map(lambda x: f"{x:.1e}")
    for col in table_df.columns:
        table_df[col] = table_df[col].map(lambda x: f"{x:.1f}")

    Path(f"tables/rebuttal_isoflops_net2net/{title}.md").parent.mkdir(parents=True, exist_ok=True)
    table_df.to_markdown(f"tables/rebuttal_isoflops_net2net/{title}.md")

## LLM PAWS


In [ ]:
import seaborn as sns
sns.set_style("white")
sns.set_context("notebook", font_scale=1.5)

ADD_BASE_COMPUTE = False
TKPM = 20.
MIN_FLOPS_SCALE_FACTOR = 6
NUM_FLOP_INTERVALS = 8
METHOD = 'paws'
SHRINK = 0.4
TARGET_RANGE_L = 2
TARGET_RANGE_R = 4

warmstarting_tkpms_df = warmstarting_df[warmstarting_df['tkpm']==TKPM]
method_tkpms_df = warmstarting_tkpms_df[warmstarting_tkpms_df['method']==METHOD]

target_models = sorted(method_tkpms_df['target_N'].unique())
target_models = target_models[TARGET_RANGE_L:TARGET_RANGE_R]  # skip the smallest model
fig, axes = plt.subplots(len(target_models), 1, figsize=(SHARED_WIDTH, SHARED_HEIGHT * len(target_models)), layout='constrained')
axes = np.atleast_1d(axes)

tables = {}
for i, target_model in enumerate(target_models):
    target_model_df = warmstarting_tkpms_df[warmstarting_tkpms_df['target_N']==target_model]
    no_growth_df = target_model_df[target_model_df['method']=='mup']

    # calculate flop intervals
    max_flops = no_growth_df['flops'].max()
    min_flops = max_flops / MIN_FLOPS_SCALE_FACTOR
    flop_intervals = np.linspace(min_flops, max_flops, NUM_FLOP_INTERVALS)
    flops_df = pd.DataFrame()

    # add growth factor == 1
    no_growth_df = no_growth_df[no_growth_df["base_N"]==no_growth_df["base_N"].max()]
    flops_df[1.] = get_loss_at_flops(no_growth_df, flop_intervals, y_col, unique_col_list)

    # add growth factor > 1
    shrink_target_model_df = target_model_df[target_model_df['method'] == METHOD]
    if SHRINK is not None:
        shrink_target_model_df = shrink_target_model_df[shrink_target_model_df['shrink']==SHRINK]
    # check if it is in a shrink list

    base_models = sorted(shrink_target_model_df['base_N'].unique(), reverse=True)
    for base_model in base_models:
        base_model_df = shrink_target_model_df[shrink_target_model_df['base_N']==base_model]

        growth_factor = target_model / base_model
        growth_df = base_model_df[base_model_df['target_N']==target_model]
        flops_df[growth_factor] = get_loss_at_flops(growth_df, flop_intervals, y_col, unique_col_list, add_base_compute=ADD_BASE_COMPUTE)
        # select only the shrink factor we want
    axes[i].set_title(f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}")
    table_df = plot_isoflops(
        axes[i],
        flops_df,
        disable_y_label=True,
        return_table=True,
    )
    tables[f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {TKPM}"] = table_df

# add figure wide xlabel
fig.supxlabel(" ")
fig.supylabel(y_col)
fig.suptitle(f"SZP - LLM")
# save the figure in a great resulution
fig.savefig(f"figures/neurips_main_{METHOD}_{TKPM}-tkpm_isoflops_gpt.pdf")

# create a markdown file for each table
for title, table_df in tables.items():
    table_df.index = table_df.index.map(lambda x: f"{x:.1e}")
    for col in table_df.columns:
        table_df[col] = table_df[col].map(lambda x: f"{x:.1f}")

    Path(f"tables/rebuttal_isoflops_net2net/{title}.md").parent.mkdir(parents=True, exist_ok=True)
    table_df.to_markdown(f"tables/rebuttal_isoflops_net2net/{title}.md")

## PAWS 2x2 -> 4

In [ ]:
import seaborn as sns
sns.set_style("white")
sns.set_context("notebook", font_scale=1.5)

ADD_BASE_COMPUTE = False
TKPM = [20., 30.]
MIN_FLOPS_SCALE_FACTOR = 6
NUM_FLOP_INTERVALS = 8
METHOD = 'paws'
SHRINK = 0.4
TARGET_RANGE_L = 2
TARGET_RANGE_R = 4


method_tkpms_df = warmstarting_df[warmstarting_df['method']==METHOD]

target_models = sorted(method_tkpms_df['target_N'].unique())
target_models = target_models[TARGET_RANGE_L:TARGET_RANGE_R]
n_images = len(target_models) * len(TKPM)
fig, axes = plt.subplots(1, n_images, figsize=(5.5 * n_images, 5) , layout='constrained')
axes = np.atleast_1d(axes)

tables = {}
for i, target_model in enumerate(target_models):
    for j, tkpm in enumerate(TKPM):
        # index it
        ax = axes[len(target_models)*i + j]
        warmstarting_tkpms_df = warmstarting_df[warmstarting_df['tkpm']==tkpm]
        target_model_df = warmstarting_tkpms_df[warmstarting_tkpms_df['target_N']==target_model]
        no_growth_df = target_model_df[target_model_df['method']=='mup']

        # calculate flop intervals
        max_flops = no_growth_df['flops'].max()
        min_flops = max_flops / MIN_FLOPS_SCALE_FACTOR
        flop_intervals = np.linspace(min_flops, max_flops, NUM_FLOP_INTERVALS)
        flops_df = pd.DataFrame()

        # add growth factor == 1
        no_growth_df = no_growth_df[no_growth_df["base_N"]==no_growth_df["base_N"].max()]
        flops_df[1.] = get_loss_at_flops(no_growth_df, flop_intervals, y_col, unique_col_list)

        # add growth factor > 1
        shrink_target_model_df = target_model_df[target_model_df['method'] == METHOD]
        if SHRINK is not None:
            shrink_target_model_df = shrink_target_model_df[shrink_target_model_df['shrink']==SHRINK]
        # check if it is in a shrink list

        base_models = sorted(shrink_target_model_df['base_N'].unique(), reverse=True)
        for base_model in base_models:
            base_model_df = shrink_target_model_df[shrink_target_model_df['base_N']==base_model]

            growth_factor = target_model / base_model
            growth_df = base_model_df[base_model_df['target_N']==target_model]
            flops_df[growth_factor] = get_loss_at_flops(growth_df, flop_intervals, y_col, unique_col_list, add_base_compute=ADD_BASE_COMPUTE)
            # select only the shrink factor we want
        ax.set_title(f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {tkpm}")
        table_df = plot_isoflops(
            ax,
            flops_df,
            disable_y_label=(i != len(target_models) - 1) or (j != len(TKPM) - 1),
            return_table=True,
        )
        tables[f"Target N: {(target_model/1_000_000):3.1f}M, tkpm: {tkpm}"] = table_df

# add figure wide xlabel
fig.supxlabel("Growth Factor")
fig.supylabel(y_col)
# save the figure in a great resulution
fig.savefig(f"figures/neurips_main_{METHOD}_{TKPM}-tkpm_isoflops_gpt.pdf")


In [ ]:
mlp_df['scaling'] == 'width'